# **Data Preprocessing**

## *Download*

Import needed modules.

In [ ]:
import requests
import os

from tqdm.notebook import tqdm

import huggingface_hub
import warnings

Define some parameters.

In [ ]:
# time interval for the dataset
YEAR: int = 2025
MONTH: int = 7

DAY_FROM: int = 1   # first day considered
DAY_TO: int = 7     # last day considered

# download
SOURCE: str = "andsanv/ais-tracks"
DOWNLOAD_PATH: str = "dataset/parquet/raw"

Download the data.

In [ ]:
# open huggingface apis
warnings.filterwarnings("ignore", message="The secret `HF_TOKEN` does not exist in your Colab secrets.")
warnings.filterwarnings("ignore", message="`local_dir_use_symlinks` parameter is deprecated")
api = huggingface_hub.HfApi()

# find parquet files
files = api.list_repo_files(SOURCE, repo_type="dataset")

# download files
for f in files:
    if f.endswith(".parquet"):
        huggingface_hub.hf_hub_download(
            repo_id=SOURCE,
            filename=f,
            repo_type="dataset",
            local_dir=DOWNLOAD_PATH,
            local_dir_use_symlinks=False
        )

## *Data Wrangling*

Import needed modules.

In [ ]:
import numpy as np
import pandas as pd
import polars as pl

Define parameters.

In [ ]:
# grid limits and cell parameters
TOP_LEFT: tuple[int, int] = (0.0, 60.0)
BOTTOM_RIGHT: tuple[int, int] = (20.0, 50.0)
CELL_DIMENSIONS: tuple[int, int] = (0.01, 0.01)   # dimensions of the single cell in the grid

Load dataframes from parquets.

In [ ]:
# create a list to store dataframes
raw_lazy_dfs: list = []

for day in tqdm(range(DAY_FROM, DAY_TO + 1), desc="Loading"):
    # lazily load dataset
    raw_lazy_df = pl.scan_parquet(f"{DOWNLOAD_PATH}/aisdk-{YEAR}-{MONTH:02d}-{day:02d}.parquet")    # load
    raw_lazy_dfs.append(raw_lazy_df)

## *Discretization*

Define some helpers for discretization.

In [ ]:
def discretize_segment(df: pl.DataFrame, cell_width: float, cell_height: float):
    """
    The method wrangles data by sampling only when a ship moves to a neighbor grid cell.
    It Calculates the 'dt' (time elapsed) since the last cell change.
    """

    out = (
        df
        .sort("timestamp")  # sort by time to ensure calculations are correct
        .with_columns([     # discretizes coordinates of the ships
            ((pl.col("x") - TOP_LEFT[0]) / cell_width).floor().cast(pl.Int32).alias("new_x"),
            ((pl.col("y") - BOTTOM_RIGHT[1]) / cell_height).floor().cast(pl.Int32).alias("new_y"),
        ])
        .with_columns([     # compare current sample's position with previous sample's position
            pl.col("new_x").shift(1).alias("prev_new_x"),
            pl.col("new_y").shift(1).alias("prev_new_y"),
        ])
        .filter(    # only keep rows if the sample is a new cell or it's the first point of the segment
            (pl.col("new_x") != pl.col("prev_new_x")) |
            (pl.col("new_y") != pl.col("prev_new_y")) |
            (pl.col("prev_new_x").is_null())
        )
        .with_columns([     # computes deltas of some attributes with respect to the previous sample
            # time
            (pl.col("timestamp") - pl.col("timestamp").shift(1))
            .dt.total_seconds()
            .fill_null(0.0) # first point has dt=0
            .cast(pl.Float32)
            .alias("dt"),

            # SOG
            (pl.col("SOG") - pl.col("SOG").shift(1))
            .fill_null(0.0)
            .cast(pl.Float32)
            .alias("dSOG"),

            # COG_sin and COG_cos
            (
                (pl.col("COG_sin") * pl.col("COG_cos").shift(1)) -
                (pl.col("COG_cos") * pl.col("COG_sin").shift(1))
            ).alias("sin_diff"),
            (
                (pl.col("COG_cos") * pl.col("COG_cos").shift(1)) +
                (pl.col("COG_sin") * pl.col("COG_sin").shift(1))
            ).alias("cos_diff")
        ])
        .with_columns([     # compute the difference angle in radians
            pl.arctan2(pl.col("sin_diff"), pl.col("cos_diff"))  # arctan2 returns a value in [-pi, pi]
            .fill_null(0.0)
            .cast(pl.Float32)
            .alias("dCOG")
        ])
        .drop(["x", "y", "prev_new_x", "prev_new_y", "sin_diff", "cos_diff"])   # remove helper columns
        .select([   # restore metadata
            pl.col("timestamp"),
            pl.col("MMSI"),
            pl.col("type"),
            pl.col("segment"),
            pl.col("new_x").alias("x"),     # change names of discretized coordinates to original coordinates' names
            pl.col("new_y").alias("y"),
            pl.col("SOG"),
            pl.col("COG_sin"),
            pl.col("COG_cos"),
            pl.col("dt"),
            pl.col("dSOG"),
            pl.col("dCOG"),
        ])
    )

    return out

Discretize the dataset by keeping only one sample per grid cell and embedding time through a new attribute.

In [ ]:
# define the output schema
output_schema = {
    "timestamp": pl.Datetime,
    "MMSI": pl.Int32,
    "type": pl.Categorical,
    "segment": pl.UInt32,
    "x": pl.Int32,
    "y": pl.Int32,
    "SOG": pl.Float32,
    "COG_sin": pl.Float32,
    "COG_cos": pl.Float32,
    "dt": pl.Float32,
    "dSOG": pl.Float32,
    "dCOG": pl.Float32,
}

# create target list and iterate through all days
lazy_dfs: list = []

for i in tqdm(range(len(raw_lazy_dfs)), desc="Discretization"):
    lazy_dfs.append(
        (
            raw_lazy_dfs[i]
            .group_by(["MMSI", "segment"])
            .map_groups(
                lambda group_df: discretize_segment(group_df, CELL_DIMENSIONS[0], CELL_DIMENSIONS[1]),
                schema=output_schema
            )
        )
    )

# **Model**

### *Prepare tensors*

Define needed modules

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from numpy.lib.stride_tricks import sliding_window_view
import gc  # garbage collector

Define some parameters

In [ ]:
#dataset dimension
days: int = DAY_TO - DAY_FROM + 1
index: int = int(days * 0.9)
NUMERIC_COLS = ["x", "y", "SOG", "COG_sin", "COG_cos", "dt", "dSOG", "dCOG"]
BATCH_SIZE = 1024

Split into training and validation set.

In [ ]:
train_lazy_dfs: list = lazy_dfs[:index]
val_lazy_dfs: list = lazy_dfs[index:]

print(f"total = {days} days\n - training: {len(train_lazy_dfs)} days\n - validation: {len(val_lazy_dfs)} days")

Define helpers.

In [ ]:
def compute_normalization_stats(dfs):
    """
    calc global max/quantiles for normalization
    """
    # combine all lazy frames
    all_data = pl.concat(dfs)

    # get stats
    s = all_data.select([
        pl.col("x").max().alias("x_max"),
        pl.col("y").max().alias("y_max"),
        pl.col("SOG").max().alias("SOG_max"),
        pl.col("dSOG").abs().quantile(0.999).alias("dSOG_scale"),
        (pl.col("dt").log1p().max()).alias("log_dt_max")
    ]).collect()

    return {
        "x_max": s["x_max"][0],
        "y_max": s["y_max"][0],
        "SOG_max": s["SOG_max"][0],
        "dSOG_scale": s["dSOG_scale"][0],
        "log_dt_max": s["log_dt_max"][0]
    }

def normalize_in_place(X, stats, is_target=False):
    # sanitize nans
    np.nan_to_num(X, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

    st = stats

    if not is_target:
        # input cols: [x, y, SOG, COG_sin, COG_cos, dt, dSOG, dCOG]
        X[..., 0] /= st["x_max"]
        X[..., 1] /= st["y_max"]
        X[..., 2] /= st["SOG_max"]
        X[..., 5] = np.log1p(X[..., 5]) / st["log_dt_max"]
        X[..., 6] /= st["dSOG_scale"]
        X[..., 7] /= np.pi
    else:
        # target cols: [dt, dSOG, dCOG, SOG, COG_sin]
        X[..., 0] = np.log1p(X[..., 0]) / st["log_dt_max"]
        X[..., 1] /= st["dSOG_scale"]
        X[..., 2] /= np.pi
        X[..., 3] /= st["SOG_max"]

    # final stability check
    np.nan_to_num(X, copy=False, nan=0.0)

def prepare_and_normalize_seq2seq_tensors(
    lazy_dfs,
    stats,
    input_len: int = 20,
    output_len: int = 20,
):
    """
    build independent encoder/decoder tensors using sliding windows
    """

    # temp storage
    lst_x, lst_t, lst_dec = [], [], []
    lst_y_c, lst_y_r = [], []

    print(f"Generating Windows (in={input_len}, out={output_len})...")

    t_map = {"Class A": 0, "Class B": 1}
    w_tot = input_len + output_len

    for lf in tqdm(lazy_dfs, desc="Processing"):
        d = lf.collect()
        if d.height == 0: continue

        # encode type
        d = d.with_columns(
            pl.col("type")
              .cast(pl.String)
              .replace_strict(t_map, default=0)
              .cast(pl.Int64)
              .alias("type_id")
        )

        # calc lookahead deltas
        d = d.with_columns([
            (pl.col("x").shift(-1) - pl.col("x")).fill_null(0).cast(pl.Int32).alias("dx"),
            (pl.col("y").shift(-1) - pl.col("y")).fill_null(0).cast(pl.Int32).alias("dy"),
        ])

        # map dy/dx to 0..8 class index
        d = d.with_columns(
            ((pl.col("dy").clip(-1, 1) + 1) * 3 + (pl.col("dx").clip(-1, 1) + 1))
            .cast(pl.Int64)
            .alias("cls")
        )

        # group by vessel id/segment
        for _, grp in d.group_by(["MMSI", "segment"]):
            grp = grp.sort("timestamp")
            if len(grp) <= w_tot: continue

            # raw arrays
            raw_num = grp.select(NUMERIC_COLS).to_numpy().astype(np.float32)
            raw_typ = grp.select("type_id").to_numpy().flatten()
            raw_cls = grp.select("cls").to_numpy().flatten()
            raw_reg = grp.select(["dt", "dSOG", "dCOG", "SOG", "COG_sin"]).to_numpy().astype(np.float32)

            # sliding windows
            # note: w_num shape becomes (N, w_tot, F)
            w_num = np.moveaxis(sliding_window_view(raw_num[:-1], w_tot, 0), -1, 1)
            w_reg = np.moveaxis(sliding_window_view(raw_reg[:-1], w_tot, 0), -1, 1)
            w_typ = sliding_window_view(raw_typ[:-1], w_tot, 0)
            w_cls = sliding_window_view(raw_cls[:-1], w_tot, 0)

            n_win = min(len(w_num), len(w_cls))

            # slice encoder (0 -> input_len)
            lst_x.append(w_num[:n_win, :input_len, :])
            lst_t.append(w_typ[:n_win, :input_len])

            # slice decoder (input_len -> end)
            lst_dec.append(w_num[:n_win, input_len:w_tot, :])

            # slice targets
            lst_y_c.append(w_cls[:n_win, input_len:w_tot])
            lst_y_r.append(w_reg[:n_win, input_len:w_tot, :])

    print("Concatenating...")

    # merge arrays
    x_all = np.concatenate(lst_x, axis=0)
    del lst_x; gc.collect()

    t_full = np.concatenate(lst_t, axis=0)
    del lst_t; gc.collect()

    # flatten type to single id per seq
    x_typ = t_full[:, 0]

    dec_all = np.concatenate(lst_dec, axis=0)
    del lst_dec; gc.collect()

    y_cls = np.concatenate(lst_y_c, axis=0)
    del lst_y_c; gc.collect()

    y_reg = np.concatenate(lst_y_r, axis=0)
    del lst_y_r; gc.collect()

    print("Normalizing...")
    normalize_in_place(x_all, stats)
    normalize_in_place(dec_all, stats)
    normalize_in_place(y_reg, stats, is_target=True)

    print(f"Final: X={x_all.shape}, Dec={dec_all.shape}, Y_cls={y_cls.shape}, Y_reg={y_reg.shape}")
    return x_all, x_typ, dec_all, y_cls, y_reg

Compute normalization statistics on the data

In [ ]:
# compute statistics on the training data (no validation data, to avoid data leakage)
norm_stats = compute_normalization_stats(train_lazy_dfs)

print("Statistics")
for k, v in norm_stats.items():
    print(f"- {k}: {v}")

Prepare the dataset into tensors with their respective targets.

In [ ]:
print("Preparing TRAIN tensors...")
X_train, types_train, Dec_train, Ycls_train, Yreg_train = prepare_and_normalize_seq2seq_tensors(
    train_lazy_dfs,
    stats=norm_stats,
    input_len=20,
    output_len=20
)

print("Preparing VAL tensors...")
X_val, types_val, Dec_val, Ycls_val, Yreg_val = prepare_and_normalize_seq2seq_tensors(
    val_lazy_dfs,
    stats=norm_stats,
    input_len=20,
    output_len=40
)

Create a wrapper class for the dataset

In [ ]:
class MaritimeSeq2SeqDataset(Dataset):
    def __init__(self, x_num, x_type, dec_num, y_cls, y_reg):
        self.x_num = torch.from_numpy(x_num).float()
        self.x_type = torch.from_numpy(x_type).long()
        self.dec_num = torch.from_numpy(dec_num).float() # Future inputs for Teacher Forcing
        self.y_cls = torch.from_numpy(y_cls).long()
        self.y_reg = torch.from_numpy(y_reg).float()

    def __len__(self):
        return len(self.x_num)

    def __getitem__(self, idx):
        return self.x_num[idx], self.x_type[idx], self.dec_num[idx], self.y_cls[idx], self.y_reg[idx]

Define datasets and loaders

In [ ]:
train_dataset = MaritimeSeq2SeqDataset(X_train, types_train, Dec_train, Ycls_train, Yreg_train)
val_dataset   = MaritimeSeq2SeqDataset(X_val,   types_val,   Dec_val,   Ycls_val,   Yreg_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### Bidirectional LSTM encoder-decoder

Import needed modules

In [ ]:
import random
import torch.nn.functional as F
import torch.optim as optim
import torch.amp as amp
import time
import csv
from google.colab import drive
import os

Implement the model

In [ ]:
class MaritimeSeq2Seq(nn.Module):
    def __init__(
        self,
        num_types: int,
        numeric_dim: int,
        norm_stats: dict,
        embed_dim: int = 2,
        hidden_dim: int = 64,
        num_layers: int = 1,
        num_classes: int = 9,
        dropout: float = 0.0,
    ):
        super().__init__()

        self.h_dim = hidden_dim
        self.n_lay = num_layers

        # normalization factors for autoregression
        self.sx = 1.0 / norm_stats['x_max']
        self.sy = 1.0 / norm_stats['y_max']

        # grid movement deltas (3x3)
        self.register_buffer('moves', torch.tensor([
            [-1, -1], [-1, 0], [-1, 1],
            [ 0, -1], [ 0, 0], [ 0, 1],
            [ 1, -1], [ 1, 0], [ 1, 1]
        ], dtype=torch.float32))

        # embedding and input projection
        self.emb_typ = nn.Embedding(num_types, embed_dim)
        dim_in = numeric_dim + embed_dim

        self.proj_in = nn.Sequential(
            nn.Linear(dim_in, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )

        # encoder (bidirectional)
        self.rnn_enc = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        # projection to map bidirectional states to decoder size
        self.bridge_h = nn.Linear(hidden_dim * 2, hidden_dim)
        self.bridge_c = nn.Linear(hidden_dim * 2, hidden_dim)

        # decoder (unidirectional)
        self.rnn_dec = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=False
        )

        # output heads
        self.out_cls = nn.Linear(hidden_dim, num_classes)

        # regression heads
        self.out_dt = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Softplus())
        self.out_sog = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())
        self.out_sin = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Tanh())
        self.out_dsog = nn.Linear(hidden_dim, 1)
        self.out_dcog = nn.Linear(hidden_dim, 1)

    def forward(
        self,
        x_numeric,
        type_id,
        target_len,
        teacher_forcing_ratio=0.5,
        future_inputs=None,
        infer_mode: str = "gumbel",
        temperature: float = 1.0
    ):
        b, t_in, _ = x_numeric.size()

        # embed type and concat with input
        t_emb = self.emb_typ(type_id) # (b, emb)
        t_seq = t_emb.unsqueeze(1).expand(b, t_in, -1)

        x_enc = self.proj_in(torch.cat([x_numeric, t_seq], dim=-1))

        # run encoder
        _, (h, c) = self.rnn_enc(x_enc)

        # reshape bidirectional states: (layers*2, b, h) -> (layers, 2, b, h)
        h = h.view(self.n_lay, 2, b, self.h_dim)
        c = c.view(self.n_lay, 2, b, self.h_dim)

        # concat directions and project
        h_cat = torch.cat([h[:, 0], h[:, 1]], dim=-1)
        c_cat = torch.cat([c[:, 0], c[:, 1]], dim=-1)

        h_dec = self.bridge_h(h_cat)
        c_dec = self.bridge_c(c_cat)

        # decoder loop setup
        preds_cls = []
        preds_reg = []
        idx_hist = []

        curr_x = x_numeric[:, -1, :].unsqueeze(1) # (b, 1, f)
        hx, cx = h_dec, c_dec

        for t in range(target_len):
            # prepare decoder input
            t_step = t_emb.unsqueeze(1)
            x_step = self.proj_in(torch.cat([curr_x, t_step], dim=-1))

            # rnn step
            out, (hx, cx) = self.rnn_dec(x_step, (hx, cx))
            feat = out.squeeze(1)

            # heads
            logits = self.out_cls(feat)

            d_t = self.out_dt(feat)
            d_sog = self.out_dsog(feat)
            d_cog = self.out_dcog(feat)
            v_sog = self.out_sog(feat)
            v_sin = self.out_sin(feat)

            reg_out = torch.cat([d_t, d_sog, d_cog, v_sog, v_sin], dim=1)

            # save step outputs
            preds_cls.append(logits.unsqueeze(1))
            preds_reg.append(reg_out.unsqueeze(1))

            # next step prep
            force = (future_inputs is not None) and (random.random() < teacher_forcing_ratio)

            if force:
                curr_x = future_inputs[:, t, :].unsqueeze(1)
            else:
                # sampling strategy
                if infer_mode == "greedy":
                    k = logits.argmax(dim=-1)
                    deltas = self.moves[k]
                elif infer_mode == "sample":
                    probs = F.softmax(logits / temperature, dim=-1)
                    k = torch.multinomial(probs, 1).squeeze(1)
                    deltas = self.moves[k]
                elif infer_mode == "gumbel":
                    soft = F.gumbel_softmax(logits, tau=temperature, hard=True)
                    deltas = torch.matmul(soft, self.moves)
                    k = soft.argmax(dim=-1)

                idx_hist.append(k)

                # manual autoregression update
                dx, dy = deltas[:, 1], deltas[:, 0]

                next_val = curr_x.clone().squeeze(1)

                # update pos
                next_val[:, 0] += dx * self.sx
                next_val[:, 1] += dy * self.sy

                # update dynamics
                next_val[:, 2] = reg_out[:, 3] # sog

                # trig fix for cog
                sin_new = reg_out[:, 4]
                cos_old = next_val[:, 4]
                cos_new = torch.sqrt((1.0 - sin_new**2).clamp(min=1e-6)) * torch.sign(cos_old)

                next_val[:, 3] = sin_new
                next_val[:, 4] = cos_new

                # update deltas
                next_val[:, 5] = reg_out[:, 0] # dt
                next_val[:, 6] = reg_out[:, 1] # dsog
                next_val[:, 7] = reg_out[:, 2] # dcog

                curr_x = next_val.unsqueeze(1)

        # combine timesteps
        seq_cls = torch.cat(preds_cls, dim=1)
        seq_reg = torch.cat(preds_reg, dim=1)

        seq_idx = torch.stack(idx_hist, dim=1) if len(idx_hist) > 0 else None

        return seq_cls, seq_reg, seq_idx

### *Training*

Define parameters

In [ ]:
NUM_TYPES_IN_DATASET = 2 # Class A and Class B
NUMERIC_DIM_IN_DATA = 8
NUM_CLASSES = 9
NUM_REGRESS = 5

# CONFIGURATION
FORCE_RESTART = False
MODEL_NAME = "BiLSTM_128Hidden"
NUM_EPOCHS = 20
VALIDATE_EVERY = 2
HIDDEN_DIM = 128
SAVE_TO_DRIVE = True

# CURRICULUM SETTINGS
TF_START = 0.8
TF_END   = 0.0

Saving the model and setup the logging

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/Maritime_Models"
    print(f"📂 Saving to Google Drive: {BASE_DIR}")
else:
    BASE_DIR = "/content/Maritime_Models_Local"
    print(f"📂 Saving locally to Runtime: {BASE_DIR}")

os.makedirs(BASE_DIR, exist_ok=True)

# define file paths
checkpoint_filename = f"maritime_model_checkpoint_{MODEL_NAME}.pth"
SAVE_PATH = os.path.join(BASE_DIR, checkpoint_filename)
LOG_PATH  = os.path.join(BASE_DIR, "training_log.csv")

# logging setup
CSV_HEADERS = [
    "Model_Name", "Batch_Size", "Current_TF", "Total_Epochs",
    "Epoch", "Time(s)", "Train_Loss", "Val_Loss", "Window_Acc(%)", "1Step_Acc(%)", "LR"
]

if not os.path.exists(LOG_PATH):
    with open(LOG_PATH, mode='w', newline='') as f:
        csv.writer(f).writerow(CSV_HEADERS)
elif FORCE_RESTART:
    with open(LOG_PATH, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([""] * len(CSV_HEADERS))
        writer.writerow(["--- RESTART ---"] + [""] * (len(CSV_HEADERS) - 1))


Define and load the model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MaritimeSeq2Seq(
    num_types=NUM_TYPES_IN_DATASET,
    numeric_dim=NUMERIC_DIM_IN_DATA,
    num_classes=NUM_CLASSES,
    hidden_dim=HIDDEN_DIM,
    norm_stats=norm_stats
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler('cuda')
criterion_cls = nn.CrossEntropyLoss()
criterion_reg = nn.MSELoss()


# checkpoint loading
start_epoch = 0
if os.path.exists(SAVE_PATH) and not FORCE_RESTART:
    print("🔄 Found checkpoint. Loading...")
    try:
        checkpoint = torch.load(SAVE_PATH, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"✅ Resumed from Epoch {start_epoch}")
    except Exception as e:
        print(f"⚠️ Load failed ({e}). Starting fresh.")
else:
    if FORCE_RESTART: print("⚠️ FORCE_RESTART is ON. Ignoring saved weights.")


if device.type == "cuda":
    model = torch.compile(model, mode="reduce-overhead")

Train the model

In [ ]:
print(f"Starting Training (Hidden={HIDDEN_DIM}, Batch={BATCH_SIZE})...")

for ep in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()
    model.train()
    sum_loss = 0.0

    # decay teacher forcing
    if NUM_EPOCHS > 1:
        tf = TF_START - (ep / (NUM_EPOCHS - 1)) * (TF_START - TF_END)
    else:
        tf = TF_START
    tf = max(0.0, min(1.0, tf))

    pbar = tqdm(train_loader, desc=f"Ep {ep+1} [TF: {tf:.2f}]", leave=False)

    for bx, bt, b_dec, by_cls, by_reg in pbar:
        # to device
        bx, bt = bx.to(device, non_blocking=True), bt.to(device, non_blocking=True)
        b_dec = b_dec.to(device, non_blocking=True)
        by_cls, by_reg = by_cls.to(device, non_blocking=True), by_reg.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            # forward pass
            out_cls, out_reg, _ = model(
                bx, bt, target_len=by_cls.size(1),
                teacher_forcing_ratio=tf, future_inputs=b_dec
            )

            loss = criterion_cls(out_cls.permute(0, 2, 1), by_cls) + criterion_reg(out_reg, by_reg)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        sum_loss += loss.item()

    # epoch stats
    train_loss = sum_loss / len(train_loader)
    dt = time.time() - t0
    lr = optimizer.param_groups[0]['lr']

    # handle torch.compile unwrapping if needed
    mod_save = model._orig_mod if hasattr(model, "_orig_mod") else model

    torch.save({
        'epoch': ep,
        'model_state_dict': mod_save.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'loss': train_loss,
    }, SAVE_PATH)

    # validation block
    msg_val, val_loss, acc_win, acc_s1 = "", None, None, None

    if (ep + 1) % VALIDATE_EVERY == 0 or (ep + 1) == NUM_EPOCHS:
        model.eval()
        v_loss, n_win, tot_win, n_s1, tot_s1 = 0.0, 0, 0, 0, 0

        with torch.no_grad():
            for vx, vt, vdec, vy_cls, vy_reg in val_loader:
                vx, vt, vdec = vx.to(device), vt.to(device), vdec.to(device)
                vy_cls, vy_reg = vy_cls.to(device), vy_reg.to(device)

                with torch.amp.autocast('cuda'):
                    pc, pr, _ = model(vx, vt, target_len=vy_cls.size(1), teacher_forcing_ratio=0.0)
                    v_loss += (criterion_cls(pc.permute(0, 2, 1), vy_cls) + criterion_reg(pr, vy_reg)).item()

                    # accuracy calc
                    preds = torch.argmax(pc, dim=2)
                    n_win += (preds == vy_cls).sum().item()
                    tot_win += vy_cls.numel()
                    n_s1 += (preds[:, 0] == vy_cls[:, 0]).sum().item()
                    tot_s1 += vy_cls.size(0)

        val_loss = v_loss / len(val_loader)
        acc_win = (n_win / tot_win) * 100.0
        acc_s1 = (n_s1 / tot_s1) * 100.0
        msg_val = f"| Val Loss: {val_loss:.4f} | Win Acc: {acc_win:.2f}% | Step One Acc: {acc_s1:.2f}"

    # log to file
    with open(LOG_PATH, mode='a', newline='') as f:
        csv.writer(f).writerow([
            MODEL_NAME, BATCH_SIZE, f"{tf:.2f}", NUM_EPOCHS, ep + 1,
            f"{dt:.0f}", f"{train_loss:.4f}",
            f"{val_loss:.4f}" if val_loss else "",
            f"{acc_win:.2f}" if acc_win else "",
            f"{acc_s1:.2f}" if acc_s1 else "",
            f"{lr:.1e}"
        ])

    print(f"Ep {ep+1}/{NUM_EPOCHS} [TF: {tf:.2f}] | Time: {dt:.0f}s | Train Loss: {train_loss:.4f} {msg_val} | Saved")

# Visualization

Import the needed modules

In [ ]:
!pip install contextily -q

In [ ]:

RELATIVE_OFFSETS = torch.tensor([
    [-1, -1], [ 0, -1], [ 1, -1],  # row 1 (dy=-1)
    [-1,  0], [ 0,  0], [ 1,  0],  # row 2 (dy=0)
    [-1,  1], [ 0,  1], [ 1,  1],  # row 3 (dy=1)
], dtype=torch.float32)

RANDOM_STATE: int = 16

## Accuracy

In [ ]:
def _reconstruct_pred_numeric_sequence(model, x_numeric, cls_seq, regs_seq):
    """
    rebuild numeric decoder inputs matching the autoregressive loop
    """
    b, n_dec = cls_seq.shape

    # grab last state from encoder
    curr = x_numeric[:, -1, :].clone()
    hist = []

    # handle attr names from previous refactor (moves/sx) or original (offsets/x_scale)
    grid = model.moves
    sc_x = model.sx
    sc_y = model.sy

    for t in range(n_dec):
        k = cls_seq[:, t]
        r = regs_seq[:, t, :]

        # class -> grid delta
        d_grid = grid[k]
        dy, dx = d_grid[:, 0], d_grid[:, 1]

        nxt = curr.clone()

        # update pos
        nxt[:, 0] += dx * sc_x
        nxt[:, 1] += dy * sc_y

        # update dynamics from regression
        # r: [dt, dsog, dcog, sog, sin]
        sog = r[:, 3]
        sin_v = r[:, 4]

        nxt[:, 2] = sog

        # maintain trig consistency
        cos_old = nxt[:, 4]
        cos_new = torch.sqrt((1.0 - sin_v**2).clamp(min=1e-6)) * torch.sign(cos_old)

        nxt[:, 3] = sin_v
        nxt[:, 4] = cos_new

        # deltas
        nxt[:, 5] = r[:, 0] # dt
        nxt[:, 6] = r[:, 1] # dsog
        nxt[:, 7] = r[:, 2] # dcog

        hist.append(nxt)
        curr = nxt

    return torch.stack(hist, dim=1)


def evaluate_seq2seq_two_cycle_40(
    model,
    data_loader,
    device="cuda",
    step_len=20,
    max_steps=40
):
    assert max_steps == 2 * step_len, "This function assumes max_steps == 2 * step_len"

    model.eval()

    # setup grid for distance calc
    if 'RELATIVE_OFFSETS' in globals():
        grid_ref = RELATIVE_OFFSETS.to(device)
    else:
        grid_ref = getattr(model, 'moves', getattr(model, 'offsets', None)).to(device)

    # metrics storage
    dist_hist = [[] for _ in range(max_steps)]
    n_corr = np.zeros(max_steps)
    n_tot = np.zeros(max_steps)

    print(f"Running TWO-CYCLE evaluation: {step_len} + {step_len} = {max_steps} steps...")

    with torch.no_grad():
        for bx, bt, _, by, _ in tqdm(data_loader):
            bx = bx.to(device)
            bt = bt.to(device)
            by = by.to(device)

            b_sz = by.size(0)
            len_true = by.size(1)
            len_use = min(max_steps, len_true)

            # cycle 1: predict first chunk
            _, r1, k1 = model(
                bx, bt, target_len=step_len,
                teacher_forcing_ratio=0.0,
                future_inputs=None,
                infer_mode="sample",
                temperature=1.0,
            )

            # rebuild numeric state for next pass
            x_pred_1 = _reconstruct_pred_numeric_sequence(model, bx, k1, r1)

            # cycle 2: predict second chunk
            # append prediction to history, keep fixed window size
            n_enc = bx.size(1)
            h_comb = torch.cat([bx, x_pred_1], dim=1)
            h_new = h_comb[:, -n_enc:, :]

            _, _, k2 = model(
                h_new, bt, target_len=step_len,
                teacher_forcing_ratio=0.0,
                future_inputs=None,
                infer_mode="sample",
                temperature=1.0,
            )

            # analysis
            k_seq = torch.cat([k1, k2], dim=1)
            k_seq = k_seq[:, :len_use]

            # calc euclidean error
            d_pred = grid_ref[k_seq]
            d_true = grid_ref[by[:, :len_use]]

            pos_pred = torch.cumsum(d_pred, dim=1)
            pos_true = torch.cumsum(d_true, dim=1)

            diffs = torch.norm(pos_pred - pos_true, dim=2) # (b, t)

            # accumulate stats
            acc_mask = (k_seq == by[:, :len_use]).cpu().numpy()
            diff_vals = diffs.cpu().numpy()

            for t in range(len_use):
                dist_hist[t].extend(diff_vals[:, t])
                n_corr[t] += acc_mask[:, t].sum()
                n_tot[t] += b_sz

    # reporting
    print(f"\n{'Step':<5} | {'Acc':<8} | {'Mean Dist':<10} | {'Var':<10} | {'Std':<10}")
    print("-" * 65)

    acc_list = []
    stats = {}

    for t in range(max_steps):
        if n_tot[t] > 0:
            acc = n_corr[t] / n_tot[t]
            acc_list.append(acc)

            arr = np.array(dist_hist[t])
            mu = arr.mean()
            v = arr.var()
            s = arr.std()

            print(f"{t+1:<5} | {acc:.2%}   | {mu:.4f}     | {v:.4f}     | {s:.4f}")

            if t == max_steps - 1:
                stats = {"mean": mu, "var": v, "std": s}

    print("-" * 65)
    print(f"Avg Accuracy:      {np.mean(acc_list):.2%}")
    if stats:
        print(f"Final Step Error:  Mean={stats['mean']:.4f}, Std={stats['std']:.4f}")
    else:
        print("Final Step Error:  No data")


evaluate_seq2seq_two_cycle_40(model, val_loader, device=device, step_len=20, max_steps=40)

## Mean distance vs temperature

Evaluating the model at different temperatures

In [ ]:
import matplotlib.pyplot as plt

TARGET_STEPS = [0, 5, 10, 19]

def scan_temperatures_step_stats(
    model,
    data_loader,
    device="cuda",
    target_steps=TARGET_STEPS,
    temp_min=0.0,
    temp_max=2.0,
    num_temps=10,
):
    model.eval()

    # handle grid offsets from global or model buffer
    if 'RELATIVE_OFFSETS' in globals():
        grid = RELATIVE_OFFSETS.to(device)
    else:
        grid = getattr(model, 'moves', getattr(model, 'offsets', None)).to(device)

    t_vals = np.linspace(temp_min, temp_max, num_temps)

    # metrics storage
    stats = {
        s: {"temps": [], "mean_dist": [], "acc": []}
        for s in target_steps
    }

    with torch.no_grad():
        for t in t_vals:
            # temp containers
            dist_hist = {s: [] for s in target_steps}
            n_corr = {s: 0 for s in target_steps}
            n_tot = {s: 0 for s in target_steps}

            for bx, bt, _, by, _ in tqdm(data_loader, desc=f"T={t:.2f}"):
                bx = bx.to(device)
                bt = bt.to(device)
                by = by.to(device)

                b = by.size(0)
                n_seq = by.size(1)

                # forward pass with sampling
                _, _, preds = model(
                    bx, bt,
                    target_len=n_seq,
                    teacher_forcing_ratio=0.0,
                    future_inputs=None,
                    infer_mode="sample",
                    temperature=t,
                )

                # map indices to physical deltas
                d_pred = grid[preds]
                d_true = grid[by]

                # integrate for positions
                pos_pred = torch.cumsum(d_pred, dim=1)
                pos_true = torch.cumsum(d_true, dim=1)

                # euclidean error
                errs = torch.norm(pos_pred - pos_true, dim=2)

                mask = (preds == by).cpu().numpy()
                val_errs = errs.cpu().numpy()

                # collect data for specific steps
                for s in target_steps:
                    if s < n_seq:
                        dist_hist[s].extend(val_errs[:, s])
                        n_corr[s] += mask[:, s].sum()
                        n_tot[s] += b

            # aggregate for this temp
            for s in target_steps:
                if n_tot[s] > 0:
                    arr = np.array(dist_hist[s])
                    mu = arr.mean()
                    acc = n_corr[s] / n_tot[s]

                    stats[s]["temps"].append(t)
                    stats[s]["mean_dist"].append(mu)
                    stats[s]["acc"].append(acc)

    return t_vals, stats


# run scan
temps, res_data = scan_temperatures_step_stats(
    model,
    val_loader,
    device=device,
    target_steps=TARGET_STEPS,
    temp_min=0.001,
    temp_max=2.0,
    num_temps=10,
)

# plotting
plt.figure(figsize=(7, 5))
for s in TARGET_STEPS:
    plt.plot(
        res_data[s]["temps"],
        res_data[s]["mean_dist"],
        marker="o",
        label=f"step {s} (index {s})",
    )

plt.xlabel("Temperature")
plt.ylabel("Mean distance")
plt.title("Mean distance vs temperature at selected steps")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Simulation

Define parameters

In [ ]:
import contextily as cx

INPUT_LEN = 20
OUTPUT_LEN = 40
N_SAMPLES_MC = 50

MARGIN: float = 0.05

Define helpers

In [ ]:
import matplotlib.pyplot as plt

def grid_to_lonlat(x_grid: int, y_grid: int, top_left: tuple, bottom_right: tuple, cell_dims: tuple) -> tuple[float, float]:
    """
    converts grid indices to longitude and latitude.
    """

    longitude: float = top_left[0] + x_grid * cell_dims[0]        # compute longitude
    latitude: float = bottom_right[1] + y_grid * cell_dims[1]     # compute latitude

    return longitude, latitude

def get_true_trajectory_from_tensor(dec_tensor, idx, norm_stats):
    """
    extracts the ground truth trajectory directly from the decoder input tensor.
    """
    # extract the future window for this sample
    future_window = dec_tensor[idx] # (output_len, features)

    # denormalize x and y (indices 0 and 1)
    # create a copy to avoid modifying the original tensor in memory
    true_x = future_window[:, 0] * norm_stats["x_max"]
    true_y = future_window[:, 1] * norm_stats["y_max"]

    # stack into (n_steps, 2)
    return np.stack([true_x, true_y], axis=1)

def monte_carlo_seq2seq(model, x_tensor, types_tensor, idx, n_steps, n_samples, device, norm_stats, temperature=1.0):
    """
    runs monte carlo simulation.
    """
    # 1. Setup Data
    seed_data = x_tensor[idx]
    type_id = types_tensor[idx]

    x_batch = torch.tensor(seed_data, dtype=torch.float32, device=device).unsqueeze(0).repeat(n_samples, 1, 1)
    type_batch = torch.tensor([type_id], dtype=torch.long, device=device).repeat(n_samples)

    # define offsets for reconstruction
    offsets_map = np.array([
        [-1, -1], [ 0, -1], [ 1, -1],
        [-1,  0], [ 0,  0], [ 1,  0],
        [-1,  1], [ 0,  1], [ 1,  1]
    ], dtype=np.float32)

    # model inference
    model.eval()
    with torch.no_grad():
        _, _, sampled_cls = model(
            x_numeric=x_batch,
            type_id=type_batch,
            target_len=n_steps,
            teacher_forcing_ratio=0.0,
            future_inputs=None,
            infer_mode="sample",
            temperature=temperature
        )
        sampled_cls = sampled_cls.cpu().numpy()

    # reconstruction
    start_x = seed_data[-1, 0] * norm_stats["x_max"]
    start_y = seed_data[-1, 1] * norm_stats["y_max"]
    start_pos = np.array([start_x, start_y])

    # map class indices to [dx, dy]
    steps_delta = offsets_map[sampled_cls]

    path_deltas = np.cumsum(steps_delta, axis=1)
    simulated_paths = path_deltas + start_pos.reshape(1, 1, 2)

    return simulated_paths

def plot_simulation(
    model, Xs, types, idx, true_path, sim_paths, n_steps, n_runs,
    window_size, norm_stats, top_left, bottom_right, cell_dims, ax
):
    """
    plots a simulation, including seed, true path and predicted path(s).
    """
    # retrieve seed and de-normalize it
    seed_norm = Xs[idx]

    # denormalize Grid X, Y
    seed_x_grid = seed_norm[:, 0] * norm_stats['x_max']
    seed_y_grid = seed_norm[:, 1] * norm_stats['y_max']

    # convert grid to lat/lon
    seed_lon, seed_lat = grid_to_lonlat(seed_x_grid, seed_y_grid, top_left, bottom_right, cell_dims)

    true_lon, true_lat = grid_to_lonlat(
        true_path[:, 0], true_path[:, 1], top_left, bottom_right, cell_dims
    )

    # flattening for hist2d
    sim_flat_x = sim_paths[:, :, 0].flatten()
    sim_flat_y = sim_paths[:, :, 1].flatten()

    sim_lon, sim_lat = grid_to_lonlat(sim_flat_x, sim_flat_y, top_left, bottom_right, cell_dims)

    # dynamic map boundaries
    all_lons = np.concatenate([seed_lon, true_lon, sim_lon])
    all_lats = np.concatenate([seed_lat, true_lat, sim_lat])

    min_lon, max_lon = all_lons.min() - MARGIN, all_lons.max() + MARGIN
    min_lat, max_lat = all_lats.min() - MARGIN, all_lats.max() + MARGIN

    # force aspect ratio
    size_lon = max_lon - min_lon
    size_lat = max_lat - min_lat
    max_size = max(size_lon, size_lat)

    center_lon = (max_lon + min_lon) / 2
    center_lat = (max_lat + min_lat) / 2

    # create bins for heatmap
    lon_bins = np.arange(min_lon, max_lon, cell_dims[0])
    lat_bins = np.arange(min_lat, max_lat, cell_dims[1])

    # plot heatmap (predictions)
    ax.hist2d(
        sim_lon, sim_lat,
        bins=[lon_bins, lat_bins],
        cmap='turbo',
        cmin=1,      # transparent where count is 0
        alpha=0.6,
        zorder=3
    )

    # plot seed
    ax.plot(seed_lon, seed_lat, color='black', linewidth=3, label='Past (Seed)', zorder=5)
    ax.scatter(seed_lon[-1], seed_lat[-1], color='black', s=80, zorder=5)

    # connect last seed point to first point of the real trajectory
    full_truth_lon = np.concatenate(([seed_lon[-1]], true_lon))
    full_truth_lat = np.concatenate(([seed_lat[-1]], true_lat))

    # plot real trajectory
    ax.plot(full_truth_lon, full_truth_lat, color='black', linewidth=3, linestyle='--', label='True Future', zorder=6)

    # formatting
    ax.set_xlim(center_lon - max_size/2, center_lon + max_size/2)
    ax.set_ylim(center_lat - max_size/2, center_lat + max_size/2)
    ax.set_aspect('equal')

    ax.legend(loc='upper right', fontsize=12, framealpha=0.9)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # add map in the background
    try:
        cx.add_basemap(
            ax,
            crs="EPSG:4326",
            source=cx.providers.CartoDB.Positron
        )
    except Exception as e:
        print(f"⚠️ Map load failed: {e}")


Compute predictions

In [ ]:
import random

# setup figure
fig, axes = plt.subplots(2, 2, figsize=(15, 15))
axes = axes.flatten()


# find 4 valid random indexes
valid_count = 0
attempts = 0
max_indices = len(X_val)
plotted_indices = []

while valid_count < 4 and attempts < 100:
    attempts += 1

    # pick random index
    idx = random.randint(0, max_indices - 1)

    # ensure uniqueness
    if idx in plotted_indices:
        continue

    # get true path directly from Dec_val tensor
    true_path = get_true_trajectory_from_tensor(Dec_val, idx, norm_stats)

    # if true path has length 0, skip
    if len(true_path) == 0:
        continue

    # run monte carlo simulation
    pred_paths = monte_carlo_seq2seq(
        model=model,
        x_tensor=X_val,
        types_tensor=types_val,
        idx=idx,
        n_steps=OUTPUT_LEN,
        n_samples=N_SAMPLES_MC,
        device=device,
        norm_stats=norm_stats,
        temperature=0.9
    )

    # plot
    plot_simulation(
        model=None, # unused in plot_simulation, passing None
        Xs=X_val,
        types=types_val,
        idx=idx,
        true_path=true_path,
        sim_paths=pred_paths,
        n_steps=OUTPUT_LEN,
        n_runs=N_SAMPLES_MC,
        window_size=INPUT_LEN,
        norm_stats=norm_stats,
        top_left=TOP_LEFT,
        bottom_right=BOTTOM_RIGHT,
        cell_dims=CELL_DIMENSIONS,
        ax=axes[valid_count]
    )

    print(f"plotted index: {idx}")
    plotted_indices.append(idx)
    valid_count += 1

plt.tight_layout()
plt.show()